# Food Delivery Analytics & Recommendation System
**Data Science CA3 · Domain: Food Delivery Industry**

**Problem.** Food-delivery platforms need to (1) understand ordering behaviour and (2) personalise restaurant suggestions to raise retention and revenue.

**Objectives**
1. Clean and explore order, restaurant and customer data (EDA).
2. Segment customers with RFM + K-Means.
3. Predict order value with Linear Regression, Random Forest and Gradient Boosting; compare them.
4. Build a hybrid recommender (collaborative filtering + cuisine affinity + rating) and evaluate it against a popularity baseline.

**Dataset.** Zomato Database (Kaggle, *anas123siddiqui/zomato-database*): `orders`, `restaurant`, `users`. Place the CSVs in `data/`.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pipeline import load_data, rfm_segments, build_models, Recommender, evaluate_recommender
sns.set_theme(style='whitegrid')
orders, rest, users, df, demo = load_data()   # stops with an error if the real CSVs are missing
print('Source:', df.attrs['quality']['source']); print('Merged shape:', df.shape); display(df.head())

## 1. Data quality: cleaning report
Duplicates, missing/invalid values, outliers (3×IQR fence – capped for modelling only), orphan IDs and column-name mismatches are handled in `pipeline.load_data` and summarised below.

In [ ]:
q=df.attrs['quality']; display(pd.Series({k:v for k,v in q.items() if not isinstance(v,list)}).to_frame('value')); print('Column mapping:',q['column_mapping'])
mv=df.isna().sum(); print('Remaining missing values:', int(mv.sum()))
sns.boxplot(x=df.sales_amount); plt.title('Order value (raw) - outliers visible'); plt.show()

## 2. Exploratory Data Analysis

In [ ]:
fig,ax=plt.subplots(2,3,figsize=(18,10))
df.dropna(subset=['order_date']).groupby(df.order_date.dt.to_period('M')).size().plot(ax=ax[0,0],marker='o',title='Orders per Month')
sns.histplot(df.sales_amount.clip(upper=df.sales_amount.quantile(.99)),bins=40,kde=True,ax=ax[0,1]).set_title('Order Value Distribution')
df.cuisine.value_counts().head(8).plot.barh(ax=ax[0,2],title='Top Cuisines'); ax[0,2].invert_yaxis()
df.restaurant_name.value_counts().head(8).plot.barh(ax=ax[1,0],title='Top Restaurants'); ax[1,0].invert_yaxis()
df.groupby('dow').size().rename(dict(enumerate(['Mon','Tue','Wed','Thu','Fri','Sat','Sun']))).plot.bar(ax=ax[1,1],title='Orders by Weekday')
num=[c for c in ['sales_quantity','sales_amount','cost','rating','rating_count','month'] if c in df]
sns.heatmap(df[num].corr(),annot=True,fmt='.2f',cmap='RdYlGn',ax=ax[1,2]); ax[1,2].set_title('Correlation')
plt.tight_layout(); plt.show()

In [ ]:
print('KEY INSIGHTS')
print(f"- {len(df):,} orders from {df.user_id.nunique():,} customers across {df.restaurant_id.nunique():,} restaurants; revenue {df.sales_amount.sum():,.0f}")
print(f"- Average order value: {df.sales_amount.mean():.1f} (median {df.sales_amount.median():.1f})")
print(f"- Most popular cuisine: {df.cuisine.value_counts().index[0]} ({df.cuisine.value_counts().iloc[0]/len(df):.0%} of orders)")
print(f"- Top 10% of customers generate {df.groupby('user_id').sales_amount.sum().nlargest(max(1,df.user_id.nunique()//10)).sum()/df.sales_amount.sum():.0%} of revenue")
for c in df.attrs['user_cats'][:2]: print(f"- Highest-ordering {c}: {df[c].value_counts().index[0]}")

## 3. Customer segmentation (RFM + K-Means)
Recency, Frequency and Monetary value are log-scaled, standardised and clustered into 4 segments.

In [ ]:
seg=rfm_segments(df); print('Silhouette by k:',seg.attrs['k_scores'],'-> chosen k =',seg.attrs['k'])
display(seg.attrs['profile'])
sns.scatterplot(data=seg,x='frequency',y='monetary',hue='segment',alpha=.6); plt.xscale('log'); plt.yscale('log'); plt.title('Customer segments'); plt.show()

## 4. Order-value prediction
Target: `sales_amount`. All models are evaluated on the same 20% hold-out set against the *original* order value. **v2 setup** = base features, raw target; **enhanced** = + train-only target encodings of customer and restaurant average value, outlier-capped log-target; plus a randomised-search tuned Gradient Boosting.

In [ ]:
res,best,imp,(yt,yp)=build_models(df)
display(res); print('Best model by RMSE:',best)
fig,ax=plt.subplots(1,2,figsize=(14,5))
imp.head(8).sort_values().plot.barh(ax=ax[0],title='Permutation Feature Importance',color='#e23744')
ax[1].scatter(yt[:1500],yp[:1500],s=8,alpha=.5); lim=[0,max(yt.max(),yp.max())]; ax[1].plot(lim,lim,'r--'); ax[1].set(title=f'Actual vs Predicted ({best})',xlabel='Actual',ylabel='Predicted')
plt.show()

## 5. Hybrid recommender
`score = w_cf·collaborative + w_cu·cuisine affinity + w_rt·smoothed rating + w_pp·popularity + w_lc·city/state affinity`.
**Evaluation:** 20% of each test user's visited restaurants are hidden; we report Precision@K, Recall@K and NDCG@K. Weights are tuned on one user group and reported on a different group, versus popularity-only, content-only and collaborative-only baselines.

In [ ]:
rec=Recommender(orders,rest); ev=evaluate_recommender(rec)
print(ev['protocol'],'| test users:',ev['users_test'],'| tuned weights:',ev['weights'])
tbl=pd.DataFrame(ev['results']).T; display(tbl.style.format('{:.2%}').highlight_max(color='#d1fae5'))
tbl[['ndcg@5','ndcg@10']].plot.barh(title='NDCG by method'); plt.show()
uid=next(u for u in rec.u_idx if rec.M[rec.u_idx[u]].nnz>=4)
recs,method=rec.recommend(uid,5); print('Recommendations for user',uid,'|',method); display(pd.DataFrame(recs))

In [ ]:
print('CONCLUSIONS (generated from the results above)')
b=res[res.model==best].iloc[0]; v2=res[res.model.str.contains('v2')].sort_values('RMSE').iloc[0]
print(f"- Best order-value model: {best} (R2={b.R2}, MAE={b.MAE}, RMSE={b.RMSE}); best v2-setup model {v2.model}: RMSE={v2.RMSE}")
r=ev['results']; k=max(ev['ks']); win=max(r,key=lambda n:r[n][f'ndcg@{k}'])
print(f"- Best recommender on NDCG@{k}: {win} ({r[win][f'ndcg@{k}']:.2%}) vs popularity-only ({r['Popularity only'][f'ndcg@{k}']:.2%})")
print('- Segments:',', '.join(seg.attrs['profile'].segment))
print('Limitations: implicit feedback only; random hold-out (not time-based); no delivery-time data.')